In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer,make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR


#### Load Data

In [29]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
train

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.300,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.920,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.500,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.200,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.930,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052
...,...,...,...,...,...,...,...,...,...,...,...,...
8518,FDF22,6.865,Low Fat,0.056783,Snack Foods,214.5218,OUT013,1987,High,Tier 3,Supermarket Type1,2778.3834
8519,FDS36,8.380,Regular,0.046982,Baking Goods,108.1570,OUT045,2002,NaN,Tier 2,Supermarket Type1,549.2850
8520,NCJ29,10.600,Low Fat,0.035186,Health and Hygiene,85.1224,OUT035,2004,Small,Tier 2,Supermarket Type1,1193.1136
8521,FDN46,7.210,Regular,0.145221,Snack Foods,103.1332,OUT018,2009,Medium,Tier 3,Supermarket Type2,1845.5976


In [30]:
x = train.drop("Item_Outlet_Sales",axis=1)
y = train['Item_Outlet_Sales']

#### Identify column types

In [31]:
num_cols = x.select_dtypes(include=['int64', 'float64']).columns
cat_cols = x.select_dtypes(include=['object']).columns

#### Preprocessing (Imputation + Scaling + OneHotEncoding)

In [32]:
num_trans = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_trans = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_trans, num_cols),
        ('cat', cat_trans, cat_cols)
    ])



####  Split Train & Validation

In [33]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

####  Define Models + Hyperparameters

In [34]:
models_and_params = {
    'Ridge': (Ridge(), {'model__alpha': [0.1, 1.0, 10.0]}),

    'Lasso': (Lasso(), {'model__alpha': [0.001, 0.01, 0.1, 1]}),

    'ElasticNet': (ElasticNet(), {'model__alpha': [0.1, 1.0],'model__l1_ratio': [0.3, 0.5, 0.7]}),

    'DecisionTree': (DecisionTreeRegressor(), {'model__max_depth': [5, 10, 15],'model__min_samples_split': [2, 5, 10]}),

    'RandomForest': (RandomForestRegressor(random_state=42), {
        'model__n_estimators': [100, 200],
        'model__max_depth': [10, 15],
        'model__min_samples_split': [2, 5] }),

    'GradientBoosting': (GradientBoostingRegressor(random_state=42), {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [3, 5]
    }),

    'SVR': (SVR(), {
        'model__kernel': ['linear', 'rbf'],
        'model__C': [0.1, 1, 10]
    }),
    
}

#### Hyperparameter Tuning with GridSearchCV

In [35]:
best_models = {}
results = []

for name, (model, params) in models_and_params.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor),('model', model)])
    
    grid = GridSearchCV(pipe, params, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
    grid.fit(x_train, y_train)
    
    best_models[name] = grid.best_estimator_
    
    preds = grid.predict(x_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mae = mean_absolute_error(y_val, preds)
    r2 = r2_score(y_val, preds)
    
    results.append({
        'Model': name,
        'Best Params': grid.best_params_,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    })

results_df = pd.DataFrame(results).sort_values(by='RMSE')
print("\nModel Performance Summary:")
print(results_df)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:658: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 75531805.27567768, tolerance: 1342432.9927816885
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:658: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 77912779.97537136, tolerance: 1362076.8224666466
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:658: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 43216907.17379475, tolerance: 1342432.9927816885
  model = cd_fast.sparse_enet_coordinate_descent(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_desce


Model Performance Summary:
              Model                                        Best Params  \
4      RandomForest  {'model__max_depth': 10, 'model__min_samples_s...   
5  GradientBoosting  {'model__learning_rate': 0.05, 'model__max_dep...   
3      DecisionTree  {'model__max_depth': 5, 'model__min_samples_sp...   
1             Lasso                                {'model__alpha': 1}   
2        ElasticNet      {'model__alpha': 0.1, 'model__l1_ratio': 0.7}   
0             Ridge                             {'model__alpha': 10.0}   
6               SVR        {'model__C': 10, 'model__kernel': 'linear'}   

          RMSE         MAE        R2  
4  1027.778948  718.383120  0.611353  
5  1028.149300  724.248609  0.611073  
3  1028.926245  722.619799  0.610485  
1  1072.204671  793.520674  0.577029  
2  1074.884133  794.690217  0.574912  
0  1085.934840  804.250401  0.566127  
6  1087.053147  785.217295  0.565233  


#### Best Model Selection

In [36]:
best_model_name = results_df.iloc[0]['Model']
best_model = best_models[best_model_name]

print(f"\n Best Model: {best_model_name}")
print(f"Parameters: {results_df.iloc[0]['Best Params']}")


 Best Model: RandomForest
Parameters: {'model__max_depth': 10, 'model__min_samples_split': 2, 'model__n_estimators': 200}


#### Predict on Test Data

In [37]:
test_preds = best_model.predict(test)
submission = pd.DataFrame({'Prediction': test_preds})
submission.to_csv("final_predictions.csv", index=False)

print("\n Predictions saved as 'final_predictions.csv'")



 Predictions saved as 'final_predictions.csv'
